# 04 — Train: MLP regression

Fits one fixed direct 24-hour scikit-learn MLP for the configured target station and evaluates it once on the test feature artifact.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview and test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins the notebook's constants. Note that the MLP's own hyperparameters are not defined here — they are written inline in the fit cell further down and documented there.

**What the imports provide**

- `MLPRegressor` — scikit-learn's fully connected feed-forward network. It accepts a 2-D target, so one estimator produces all 24 horizons from one output layer.
- `StandardScaler` — used twice below, once for the predictors and once for the targets.
- `mean_absolute_error`, `root_mean_squared_error` — the two reported error metrics.
- `feature_column_names()`, `target_column_names()` — the Stage-3 column contract.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 feature Parquets are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | How many scored test rows the final preview table shows. Display only; it has no effect on any metric. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract, read from `feature_column_names()` instead of being hardcoded so the notebook fails loudly if Stage 3 ever changes it. It contains: the raw `water_level`, `imputed`, `precipitation` and `temperature_2m`; 8 water-level lags (1, 3, 6, 12, 24, 48, 72, 168 h); 5 water-level differences (1–24 h); 16 rolling water-level statistics (mean/std/min/max x 6/24/72/168 h); 4 rolling imputation counts; 4 rolling precipitation sums; 4 rolling temperature means plus the 24 h temperature min and max; and 6 calendar Fourier terms. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | The 24 future water levels, one per lead hour. The model emits all of them from a single feature vector — a *direct* multi-horizon setup, with no recursive feeding of its own predictions. |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

from src.config import TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())

## Shared evaluation cohort

Every stage-4 candidate is fit and scored on exactly the same rows, which is what makes their reported numbers comparable to each other and to the persistence baseline. One row is one *issue time* `t`, and it qualifies only when both of these hold:

1. **Stage 3 marked it `target_valid`.** All 24 future water levels `t+1 … t+24` were actually observed, none of them synthesised. This drops issue times sitting near a data gap, plus the final 24 hours of each artifact, which have no complete future.
2. **All 53 predictors are present.** The lag and rolling features need a complete 168-hour lookback, so the first week of each artifact is a warm-up that can never qualify.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it.

## Helper functions

Three small helpers used by the load / fit / evaluate cells below. All of them take keyword-only `station_id` and `artifact_name` arguments that exist purely so a raised error names the split it came from instead of leaving you to guess.

**`eligible_rows(frame, *, station_id, artifact_name) -> pd.Series`**

- `frame` — one loaded feature artifact (train or test).
- `station_id` — the station the artifact is supposed to describe; used in error messages.
- `artifact_name` — `"train"` or `"test"`, likewise for error messages.

Returns a boolean mask marking the cohort rows defined above. It raises instead of returning a mask when the artifact is missing a contract column, or when a `target_valid` row still carries a null target — that combination is a Stage-3 bug, and silently averaging around it would produce a metric that looks fine and is not.

**`metric_tables(actual, predictions, *, station_id) -> (aggregate, per_horizon)`**

- `actual` — the cohort's `TARGET_COLUMNS` frame, shape `(n_issue_times, 24)`.
- `predictions` — the model's output, in the same shape and the same row order.

Returns two frames. The **aggregate** one pools all `n x 24` values into a single MAE and RMSE. The **per-horizon** one repeats that calculation separately for each lead time, which is what reveals how fast accuracy decays from `t+1` to `t+24`. Both metrics are in the water level's native unit: MAE is the average absolute miss, while RMSE squares the errors before averaging, so it is always at least as large as MAE and is dominated by the worst forecasts — a wide gap between the two means a few large misses rather than uniformly poor accuracy.

**`prediction_preview(frame, predictions) -> pd.DataFrame`**

- `frame` — the scored cohort rows, supplying `timestamp` and the actual targets.
- `predictions` — the matching predicted array.

Returns one frame with each `prediction_target_t_plus_XX` column beside its actual counterpart, aligned on the cohort's original index so no row silently shifts.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon

In [ ]:
def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load feature artifacts

Resolves `data/processed/<station>_train_features.parquet` and `<station>_test_features.parquet` for the station configured in `src.config.TARGET_STATION_ID`, then raises `FileNotFoundError` if either is absent. A missing artifact means stages 1–3 have not been run for this station, and failing here costs a second rather than failing halfway through a fit.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)

## Apply the eligibility cohort

Builds the train and test masks with `eligible_rows()` and keeps only the rows that pass. If either split ends up with no eligible row the notebook stops here, rather than fitting on an empty frame and reporting a metric computed from nothing.

In [ ]:
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

train_rows = train_features.loc[train_mask]
test_rows = test_features.loc[test_mask]

## Standardize and fit the direct multi-output MLP

Two separate `StandardScaler` instances are fitted **on the training cohort only**:

- the *predictor* scaler, because a network trained by gradient descent converges badly when its 53 inputs span wildly different numeric ranges — large-magnitude inputs dominate the early gradients regardless of how informative they are;
- the *target* scaler, because the 24 outputs are water levels far from zero with a shared offset. Squared-error loss on raw levels is dominated by that offset, and the network would spend its first epochs learning a constant.

Test predictors go through `transform` (never `fit_transform`), and the predictions are put back through `inverse_transform` before scoring, so every reported metric is in the original water-level unit rather than in standard deviations.

**`MLPRegressor` parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `hidden_layer_sizes` | `(128, 64)` | Two hidden layers of 128 and 64 units, giving a 53 → 128 → 64 → 24 network. The taper is a conventional default shape, not a tuned choice. |
| `activation` | `"relu"` | Rectified linear units on both hidden layers — the standard non-linearity; without one the whole network would collapse into a linear model. |
| `solver` | `"adam"` | Mini-batch stochastic optimiser with per-parameter adaptive step sizes. It is the right default at this data size: `"lbfgs"` is full-batch and only competitive on very small datasets, and `"sgd"` needs a hand-tuned learning-rate schedule. |
| `alpha` | `1e-4` | L2 penalty on the weights (scikit-learn's default). It is the *only* regulariser in this configuration — there is no dropout and, see below, no early stopping. |
| `batch_size` | `256` | Training samples per gradient update. Larger batches give smoother, less noisy gradient estimates and use the CPU better; smaller ones add regularising noise. |
| `learning_rate_init` | `1e-3` | Adam's initial step size, which is also its effective step size throughout, because `learning_rate` is left at its default `"constant"`. |
| `max_iter` | `300` | Maximum number of epochs (full passes over the training data). Training stops earlier only if Adam's loss improvement stays below `tol` (`1e-4`) for `n_iter_no_change` (10) consecutive epochs. `mlp.n_iter_`, shown in the results table, tells you which of the two actually happened — a value of exactly 300 means the budget ran out and the model may still have been improving. |
| `random_state` | `42` | Seeds weight initialisation and batch shuffling so the run reproduces. |
| `early_stopping` | `False` | Deliberately off. Setting it to `True` makes scikit-learn hold out a *random* 10% of the training rows as a validation set, which would break the strictly chronological setup this pipeline depends on. The cost of leaving it off is that nothing monitors generalisation during training at all. |

In [ ]:
predictor_scaler = StandardScaler()
target_scaler = StandardScaler()
train_predictors = predictor_scaler.fit_transform(train_rows[FEATURE_COLUMNS])
test_predictors = predictor_scaler.transform(test_rows[FEATURE_COLUMNS])
train_targets = target_scaler.fit_transform(train_rows[TARGET_COLUMNS])

mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=300,
    random_state=42,
    early_stopping=False,
)
mlp.fit(train_predictors, train_targets)
test_predictions = target_scaler.inverse_transform(mlp.predict(test_predictors))

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort: aggregate MAE/RMSE, the same two metrics per lead time, and a short preview so the predictions can be eyeballed against their actual targets. There is no second pass and no refitting — what is printed here is the notebook's one and only result.

The extra `model_info` table reports `mlp.n_iter_` (epochs actually run — compare against `max_iter=300`) and `mlp.loss_` (the final training loss on *scaled* targets, so it is unitless and only comparable across runs of this same notebook).

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
model_info = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "n_iter": mlp.n_iter_,
            "loss": mlp.loss_,
        }
    ]
)
print(f"MLP test results for {station_id}")
display(model_info)
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))